# Commodity Futures Trading Strategy Development - Starter Notebook

This notebook provides basic data loading functionality to help you get started with the assessment.

## Important Notes
- Focus on methodology over pure performance
- Document your assumptions and approach
- Feel free to modify this notebook or create your own from scratch
- All data is synthetically generated and has no relation to actual market prices


In [ ]:
%matplotlib inline
import os
os.environ['PY3_PROD'] = '1'
%load_ext autoreload
%autoreload 2
os.system('kinit')

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')
import re
# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")


In [ ]:
from pycmqlib3.utility import dbaccess, dataseries, misc
from pycmqlib3.analytics.tstool import *
from pycmqlib3.analytics.btmetrics import *
from pycmqlib3.analytics.backtest_utils import *
from pycmqlib3.strategy.signal_repo import *

## 1. Load Data

Load all the provided datasets. Adjust the path if you're running this from a different directory.


In [ ]:
# Define data directory
DATA_DIR = './data/'

# Load main datasets
print("Loading price data...")
price_data = pd.read_csv(DATA_DIR + 'price_data.csv')
price_data['date'] = pd.to_datetime(price_data['date'])
print(f"Price data shape: {price_data.shape}")

print("\nLoading commodity information...")
commodity_info = pd.read_csv(DATA_DIR + 'commodity_info.csv')
print(f"Commodity info shape: {commodity_info.shape}")

print("\nLoading sector information...")
sector_info = pd.read_csv(DATA_DIR + 'sector_info.csv')
print(f"Sector info shape: {sector_info.shape}")

print("\nLoading FX data...")
fx_data = pd.read_csv(DATA_DIR + 'fx_data.csv')
fx_data['date'] = pd.to_datetime(fx_data['date'])
print(f"FX data shape: {fx_data.shape}")

# Load alternative data - economic indicators
print("\nLoading economic data...")
try:
    economic_weekly = pd.read_csv(DATA_DIR + 'economic_weekly.csv')
    economic_weekly['date'] = pd.to_datetime(economic_weekly['date'])
    print(f"Economic weekly shape: {economic_weekly.shape}")
    
    economic_monthly = pd.read_csv(DATA_DIR + 'economic_monthly.csv')
    economic_monthly['date'] = pd.to_datetime(economic_monthly['date'])
    print(f"Economic monthly shape: {economic_monthly.shape}")
except:
    print("Some economic data files not found")

# Load other alternative data
print("\nLoading other alternative data...")
alt_data_files = [
    'weather_weekly', 'supply_chain_weekly', 'geopolitical_weekly',
    'sentiment_daily', 'shipping_weekly', 'energy_weekly',
    'agricultural_weekly', 'industrial_weekly', 'financial_daily'
]

alt_data = {}
for file_name in alt_data_files:
    try:
        df = pd.read_csv(DATA_DIR + file_name + '.csv')
        df['date'] = pd.to_datetime(df['date'])
        alt_data[file_name] = df
        print(f"  {file_name}: {df.shape}")
    except:
        print(f"  {file_name}: not found")

print("\n✅ Data loading complete!")


In [ ]:
df_list = []
for tmp_data in [economic_weekly, economic_monthly, fx_data] + [alt_data[key].set_index('date') for key in alt_data.keys()]:
    if 'date' in tmp_data.columns:
        tmp_data = tmp_data.set_index('date')
    df_list.append(tmp_data)
spot_df = pd.concat(df_list, axis=1)

## 2. Basic Data Exploration

Here are some helper functions and initial explorations to get you started.


In [ ]:
# Display basic information about commodities
print("Commodity Information Summary:")
print("-" * 50)
print(f"Total commodities: {commodity_info.shape[0]}")
print(f"US commodities: {(commodity_info['market'] == 'US').sum()}")
print(f"China commodities: {(commodity_info['market'] == 'CN').sum()}")
print(f"\nNumber of sectors: {commodity_info['sector'].nunique()}")
print(f"\nSectors:")
print(commodity_info['sector'].value_counts())
print(f"\nExpiry schemes:")
print(commodity_info['expiry_scheme'].value_counts())

# Display first few rows
commodity_info.head()


In [ ]:
# Check available contracts for each commodity
print("Available contracts in the data:")
contracts = price_data['contract'].unique()
print(f"Contract types: {sorted(contracts)}")

# Check date range
print(f"\nDate range: {price_data['date'].min()} to {price_data['date'].max()}")
print(f"Number of trading days: {price_data['date'].nunique()}")


## 3. Data Access Functions

Basic functions to help you load specific commodity data.


In [ ]:
def get_commodity_data(commodity_name, contract='F1'):
    """
    Get price data for a specific commodity and contract.
    
    Parameters:
    -----------
    commodity_name : str
        The commodity name (e.g., 'US_Crude', 'CN_Gold')
    contract : str
        The contract type (e.g., 'SPOT', 'F1', 'F2', etc.)
    
    Returns:
    --------
    DataFrame with date, price, and volume
    """
    mask = (price_data['commodity'] == commodity_name) & (price_data['contract'] == contract)
    return price_data[mask][['date', 'price', 'volume']].sort_values('date').reset_index(drop=True)


def get_all_commodities():
    """
    Get list of all available commodities.
    
    Returns:
    --------
    List of commodity names
    """
    return price_data['commodity'].unique().tolist()


def get_contracts_for_commodity(commodity_name):
    """
    Get all available contracts for a specific commodity.
    
    Parameters:
    -----------
    commodity_name : str
        The commodity name
    
    Returns:
    --------
    List of contract names
    """
    return price_data[price_data['commodity'] == commodity_name]['contract'].unique().tolist()

print("Data access functions loaded successfully!")


## 4. Example: Plotting a Commodity Price Series

Here's a simple example of how to visualize commodity data.


In [ ]:
# Example: Plot price series for US Crude
example_commodity = 'US_Crude'
example_data = get_commodity_data(example_commodity, 'F1')

if not example_data.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(example_data['date'], example_data['price'])
    plt.title(f"Price Series for {example_commodity} (F1 Contract)")
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    # Check for missing data
    missing_pct = example_data['price'].isna().sum() / len(example_data) * 100
    print(f"Missing data: {missing_pct:.2f}%")
else:
    print(f"No data found for {example_commodity}")


## 5. Your Analysis Starts Here

Build your commodity futures trading strategy using the provided data.

Good luck!


In [ ]:
px_pivot = pd.pivot_table(price_data, index='date', columns=['commodity', 'contract'], values='price', aggfunc='last')
volume_pivot = pd.pivot_table(price_data, index='date', columns=['commodity', 'contract'], values='volume', aggfunc='last')

# px_pivot.columns.get_level_values(0).unique()

us_assets = [
    'US_Aluminum', 'US_Cocoa', 'US_Coffee',
    'US_Copper', 'US_Corn', 'US_Cotton', 'US_Crude', 'US_Ethanol',
    'US_Gasoline', 'US_Gold', 'US_HeatingOil', 'US_IronOre',
    'US_NaturalGas', 'US_Palladium', 'US_Platinum', 'US_Rice', 'US_Silver',
    'US_Soybeans', 'US_Sugar', 'US_Wheat'
]

cn_assets = [ asset for asset in px_pivot.columns.get_level_values(0).unique() if asset not in us_assets]

cn_index = px_pivot.loc[:, px_pivot.columns.get_level_values(0).isin(cn_assets) & px_pivot.columns.get_level_values(1).isin(["F1"])].dropna(how='all').index
us_index = px_pivot.loc[:, px_pivot.columns.get_level_values(0).isin(us_assets) & px_pivot.columns.get_level_values(1).isin(["F1"])].dropna(how='all').index


In [ ]:
def build_roll_adj_px(df, expiry_dates, roll_day=0, mode='ret', contract=1):
    curr_cont = f"F{contract}"
    next_cont = f"F{contract+1}"
    df = df.sort_index()
    index = df.index
    if mode == 'ret':
        df = np.log(df)
    expiry_dates = pd.to_datetime(expiry_dates)
    aligned_expiry, aligned_roll = [], []
    for ed in expiry_dates:
        if ed < index[0]:
            continue
        elif ed > index[-1]:
            ed_aligned = ed
        else:
            ed_aligned = index[index <= ed].max()
        rd_nominal = ed_aligned - pd.tseries.offsets.BDay(roll_day)
        if rd_nominal > index[-1]:
            continue
        elif rd_nominal < index[0]:
            rd_aligned = rd_nominal
        else:
            rd_aligned = index[index <= rd_nominal].max()
        aligned_expiry.append(ed_aligned)
        aligned_roll.append(rd_aligned)
    if not aligned_expiry:
        raise ValueError("No expiry dates fall within the data index range")

    roll_diffs = []
    for rd in aligned_roll:
        if rd in df.index:
            diff = df.loc[rd, curr_cont] - df.loc[rd, next_cont]
            roll_diffs.append((rd, diff))
        else:
            roll_diffs.append((rd, 0))

    roll_df = pd.DataFrame(roll_diffs, columns=['roll_date', 'diff']).set_index('roll_date')
    adj_series_daily = roll_df['diff'].reindex(index=index, fill_value=0).shift(1).fillna(0).cumsum()

    unadj = df[curr_cont].copy()

    if roll_day > 0:
        for ed, rd in zip(aligned_expiry, aligned_roll):
            mask = (index > rd) & (index <= ed)
            unadj.loc[mask] += df.loc[mask, next_cont] - df.loc[mask, curr_cont]

    ts = unadj + adj_series_daily
    if mode == 'ret':
        ts = np.exp(ts)
        unadj = np.exp(unadj)
        adj_series_daily = np.exp(adj_series_daily)
    ts.name = curr_cont
    # debug = {"cont": unadj, 
    #          "adj": adj_series_daily,
    #          "roll_df": roll_df}
    return ts



In [ ]:
px_dict = {}
for asset in us_assets + cn_assets:
    ratio_f1 = px_pivot[(asset, "F1")]/px_pivot[(asset, "F1")].shift()
    ratio_f2 = px_pivot[(asset, "F2")]/px_pivot[(asset, "F2")].shift()
    ratio_f3 = px_pivot[(asset, "F3")]/px_pivot[(asset, "F3")].shift()

    mask1 = px_pivot[(asset, "F1")].isna() & (~ px_pivot[(asset, "F2")].isna()) & (~ px_pivot[(asset, "F2")].shift().isna()) & (~px_pivot[(asset, "F1")].shift().isna())
    px_pivot.loc[mask1, (asset, "F1")] = px_pivot[(asset, "F1")].shift()[mask1] * ratio_f2[mask1]

    mask2 = px_pivot[(asset, "F1")].isna() & (~ px_pivot[(asset, "F3")].isna()) & (~ px_pivot[(asset, "F3")].shift().isna()) & (~px_pivot[(asset, "F1")].shift().isna())
    px_pivot.loc[mask2, (asset, "F1")] = px_pivot[(asset, "F1")].shift()[mask2] * ratio_f3[mask2]

    mask3 = px_pivot[(asset, "F2")].isna() & (~ px_pivot[(asset, "F1")].isna()) & (~ px_pivot[(asset, "F1")].shift().isna()) & (~px_pivot[(asset, "F2")].shift().isna())
    px_pivot.loc[mask3, (asset, "F2")] = px_pivot[(asset, "F2")].shift()[mask3] * ratio_f1[mask3]
    
    mask4 = px_pivot[(asset, "F3")].isna() & (~ px_pivot[(asset, "F1")].isna()) & (~ px_pivot[(asset, "F1")].shift().isna()) & (~px_pivot[(asset, "F2")].shift().isna())
    px_pivot.loc[mask4, (asset, "F3")] = px_pivot[(asset, "F3")].shift()[mask4] * ratio_f1[mask4]

    mask3 = abs(px_pivot[(asset, "SPOT")]/px_pivot[(asset, "F1")]-1) > 0.1
    px_pivot.loc[mask3, (asset, "SPOT")] = np.nan

    df = px_pivot.loc[:, px_pivot.columns.get_level_values(0)==asset].droplevel([0], axis=1)

    # # fix NA for F1, F2 mutually by the ratio
    # ratio_f2 = df["F2"]/df["F2"].shift()
    # ratio_f1 = df["F1"]/df["F1"].shift()

    # mask1 = df["F1"].isna() & (~ df["F2"].isna()) & (~ df["F2"].shift().isna()) & (~df["F1"].shift().isna())
    # df.loc[mask1, "F1"] = df["F1"].shift() * ratio_f2

    # mask2 = df["F2"].isna() & (~ df["F1"].isna()) & (~ df["F1"].shift().isna()) & (~df["F2"].shift().isna())
    # df.loc[mask2, "F2"] = df["F2"].shift() * ratio_f1
    # mask3 = abs(df["SPOT"]/df["F1"]-1) > 0.1
    # df.loc[mask3, "SPOT"] = np.nan
     
    asset_info = commodity_info[commodity_info['commodity'] == asset].to_dict("records")[0]
    if asset_info['expiry_scheme'] == 'monthly':
        mth_list = range(1, 13)
    elif asset_info['expiry_scheme'] == 'quarterly':
        mth_list = [3, 6, 9, 12]
    elif asset_info['expiry_scheme'] == 'bi-annual':
        mth_list = [6, 12]
    elif asset_info['expiry_scheme'] == 'bi-monthly':
        mth_list = [1, 3, 5, 7, 9, 11]
    else:
        mth_list = range(1, 13)
    
    roll_day = 5
    expiry_day = asset_info['expiry_day']
    
    expiry_dates = [ d + pd.DateOffset(days = expiry_day-1) for d in pd.date_range(start=df.index[0], end=df.index[-1] + pd.DateOffset(months=6), freq="MS") 
                    if d.month in mth_list]

    px_dict[asset] = build_roll_adj_px(df, expiry_dates, roll_day=roll_day, mode='ret', contract=1)


In [ ]:
import pickle
data_file = "C:/dev/data/optiver_processed.pkl"
with open(data_file, 'wb') as f:
    df_dict = {
        'hist_data': px_pivot,
        'roll_adj': px_dict,
    }
    pickle.dump(df_dict, f)

In [ ]:
# px_dict = {}
sector_assets = {}
for sector in commodity_info['sector'].unique():
    sector_assets[sector] = list(commodity_info[commodity_info['sector']==sector]['commodity'])
    f1_px = px_pivot.loc[:, (px_pivot.columns.get_level_values(0).isin(sector_assets[sector])) & (px_pivot.columns.get_level_values(1)=='F1')].droplevel([1], axis=1)
    f1_px = np.log(f1_px.dropna())
    f1_px = f1_px.diff()
    iplot(np.exp(f1_px.cumsum()), title=sector)
    display(f1_px.corr())

In [ ]:
f1_px = px_pivot.loc[:, px_pivot.columns.get_level_values(1)=='F1'].droplevel([1], axis=1)
spot_px = px_pivot.loc[:, px_pivot.columns.get_level_values(1)=='SPOT'].droplevel([1], axis=1)

In [ ]:
import matplotlib.cm as cm
start_d = pd.to_datetime('2018-09-30')
end_d = pd.to_datetime('2025-09-30')

log_f1 = np.log(f1_px.dropna(how='all').ffill())
logret = log_f1.diff()[(log_f1.index >= start_d) & (log_f1.index<=end_d)]

corr = logret.corr()
size = 10
fig, ax = plt.subplots(figsize=(size, size))
ax.matshow(corr,cmap=cm.get_cmap('coolwarm'), vmin=0,vmax=1)
plt.xticks(range(len(corr.columns)), corr.columns, rotation='vertical', fontsize=8)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=8)
plt.show()

In [ ]:
import scipy.cluster.hierarchy as sch 
from scipy.spatial.distance import pdist
import pylab

Z = sch.linkage(corr, 'ward')
print(Z[0])

# c, coph_dists = sch.cophenet(Z, pdist(corr))
# print(c)
plt.figure(figsize=(25, 10))
labelsize=20
ticksize=15
plt.title('Hierarchical Clustering Dendrogram for China Futures', fontsize=labelsize)
plt.xlabel('products', fontsize=labelsize)
plt.ylabel('distance', fontsize=labelsize)
sch.dendrogram(
    Z,
    leaf_rotation=90.,  # rotates the x axis labels
    leaf_font_size=8.,  # font size for the x axis labels
    labels = corr.columns
)
pylab.yticks(fontsize=ticksize)
pylab.xticks(rotation=-90, fontsize=ticksize)
plt.savefig('dendogram_'+'_fut'+'.png')
plt.show()

In [ ]:
vol_win=20
pnl_tenors = ['6m', '1y', '2y', '3y', '4y', '5y', '6y', '7y', '8y', '9y', '10y']
insample = "2026-01-01"

empiric_assets = cn_assets + us_assets
cont = "F1"
df_pxchg = pd.DataFrame(index=px_pivot.index)
vol_df = pd.DataFrame(index=px_pivot.index)

for asset in empiric_assets:
    # if '_' in asset:
    #     df_pxchg[asset] = beta_ret_dict[asset].dropna()[:insample]
    # elif ('-' in asset) or ('=' in asset):
    #     for sub_asset in re.split("[-=]", asset):
    #         if sub_asset not in empiric_assets:
    #             df_pxchg[sub_asset] = df[(sub_asset+'c1', traded_price)].dropna().pct_change()[:insample]
    # else:
    df_pxchg[asset] = px_pivot[(asset, cont)].dropna().pct_change() 
    vol_df = df_pxchg.rolling(vol_win).std()

df_pxchg = df_pxchg[:insample]
vol_df = vol_df[:insample]

In [ ]:
cdates = pd.date_range(start='2018-01-01', end='2025-09-30', freq='d')

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.api import OLS, add_constant
from scipy.stats import spearmanr, pearsonr

def compute_corr(x, y, method="spearman"):
    """Compute correlation and p-value safely."""
    mask = x.notna() & y.notna()
    if mask.sum() < 10:
        return np.nan, np.nan, mask.sum()
    x, y = x[mask], y[mask]
    if method == "spearman":
        c, p = spearmanr(x, y)
    else:
        c, p = pearsonr(x, y)
    return c, p


def rolling_beta(y, x, window=60):
    """
    Compute rolling regression beta of y on x.
    Returns Series of betas aligned with y.index.
    """
    betas = pd.Series(index=y.index, dtype=float)
    for i in range(window, len(y)):
        y_sub = y.iloc[i-window:i]
        x_sub = x.iloc[i-window:i]
        if y_sub.isna().any() or x_sub.isna().any():
            continue
        model = OLS(y_sub, add_constant(x_sub)).fit()
        betas.iloc[i] = model.params[1] if len(model.params) > 1 else np.nan
    return betas


def compute_forward_returns(px_df, horizons=(1, 3, 5), shift=0):
    """
    Compute forward returns starting from next day (t+1) to (t+h).
    """
    fwd_returns = {}
    for h in horizons:
        fwd_returns[h] = px_df.shift(-shift - h) / px_df.shift(-shift) - 1
    return fwd_returns


def build_feature_return_panel(px_df: pd.DataFrame,
                               spot_df: pd.DataFrame,
                               feature_name: str,
                               forward_windows=(1, 3, 5),
                               beta_window=60):
    """
    Given price DataFrame (cols = assets) and feature Series (index = dates),
    aligns them, computes forward returns at feature dates,
    and builds a long panel with [date, asset, feature, fwd_return_h].
    
    If feature_name is common (in spot_df.columns), compute rolling beta
    of each asset's return to that feature, and use beta * feature as score.

    Returns:
        panel DataFrame with columns:
        ['date', 'asset', 'feature', 'score', 'horizon', 'forward_return']
    """
    assets = list(px_df.columns)

    # --- Identify feature type ---
    if feature_name in spot_df:
        feature_type = "common"
        feature_ts = spot_df[feature_name].dropna()
    else:
        feature_type = "asset_specific"
        feature_ts = spot_df[[f"{asset}_{feature_name}" for asset in px_df.columns]].dropna(how='all')

    # --- Align data ---
    px_aligned = px_df.reindex(index=feature_ts.index, method='ffill').sort_index()
    ret_df = px_aligned.pct_change()

    # --- Forward returns ---
    fwd_returns = {}
    for h in forward_windows:
        fwd_returns[h] = px_aligned.shift(-h) / px_aligned - 1.0
    fwd_df = pd.concat(fwd_returns, axis=1)  # MultiIndex columns: (horizon, asset)

    # --- Rolling beta computation for common feature ---
    if feature_type == "common" and beta_window > 0:
        f = feature_ts.loc[ret_df.index]
        f_mean = f.rolling(beta_window).mean()
        f_var = f.rolling(beta_window).var()

        betas = pd.DataFrame(index=ret_df.index, columns=assets, dtype=float)
        for asset in assets:
            r = ret_df[asset]
            cov_rf = (r * f).rolling(beta_window).mean() - r.rolling(beta_window).mean() * f_mean
            betas[asset] = cov_rf / f_var

        # Compute score = beta * feature value
        score_df = betas.mul(f, axis=0)

    # --- Build panel ---
    panels = []
    for h in forward_windows:
        fwd_tmp = fwd_df[h].stack().rename("forward_return").reset_index()
        fwd_tmp["horizon"] = h

        if feature_type == "common":
            fwd_tmp["feature"] = feature_ts.reindex(fwd_tmp["date"]).values
            if beta_window > 0:
                fwd_tmp["beta"] = [
                    score_df.at[dt, asset] / feature_ts.at[dt] if dt in score_df.index and feature_ts.at[dt] != 0 else np.nan
                    for dt, asset in zip(fwd_tmp["date"], fwd_tmp["asset"])
                ]
                fwd_tmp["score"] = [
                    score_df.at[dt, asset] if dt in score_df.index else np.nan
                    for dt, asset in zip(fwd_tmp["date"], fwd_tmp["asset"])
                ]
            else:
                fwd_tmp["beta"] = np.nan
                fwd_tmp["score"] = fwd_tmp["feature"]

        else:
            # Asset-specific feature
            fwd_tmp["feature"] = [
                feature_ts.at[dt, asset] if dt in feature_ts.index else np.nan
                for dt, asset in zip(fwd_tmp["date"], fwd_tmp["asset"])
            ]
            fwd_tmp["score"] = fwd_tmp["feature"]
            fwd_tmp["beta"] = np.nan

        panels.append(fwd_tmp)

    panel = pd.concat(panels, ignore_index=True)
    panel = panel.dropna(subset=["score", "forward_return"])
    return panel


def analyze_feature_ic_ir(panel: pd.DataFrame, method="spearman"):
    """
    Analyze feature predictive power via Information Coefficient (IC) and Information Ratio (IR).

    Parameters
    ----------
    panel : pd.DataFrame
        Must contain ['date', 'asset', 'feature', 'forward_return', 'horizon'].
    method : str
        'spearman' (rank IC) or 'pearson' (linear IC).

    Returns
    -------
    results : dict of DataFrames
        {
            'summary': IC/IR summary per horizon,
            'per_asset_ic': IC per asset per horizon,
            'per_date_ic': IC per date per horizon
        }
    """
    results = []
    per_asset_list = []
    per_date_list = []

    for h, dfh in panel.groupby("horizon"):
        # === (1) IC per asset (time-series ICs)
        asset_corrs, asset_pvals = [], []
        for asset, sub in dfh.groupby("asset"):
            c, p = compute_corr(sub["score"], sub["forward_return"], method)
            asset_corrs.append({"asset": asset, "horizon": h, "ic": c, "pval": p})
        per_asset_df = pd.DataFrame(asset_corrs)
        per_asset_list.append(per_asset_df)

        # === (2) IC per date (cross-sectional ICs)
        date_corrs = []
        for date, sub in dfh.groupby("date"):
            c, p = compute_corr(sub["score"], sub["forward_return"], method)
            date_corrs.append({"date": date, "horizon": h, "ic": c, "pval": p})
        per_date_df = pd.DataFrame(date_corrs)
        per_date_list.append(per_date_df)

        # === (3) Compute IR stats from cross-sectional ICs (per-date)
        mean_ic = per_date_df["ic"].mean()
        std_ic = per_date_df["ic"].std()
        ir = mean_ic / std_ic if std_ic > 0 else np.nan

        results.append({
            "horizon": h,
            "mean_ic": mean_ic,
            "median_ic": per_date_df["ic"].median(),
            "ic_std": std_ic,
            "ic_ir": ir,
            "n_dates": per_date_df["ic"].notna().sum(),
            "n_assets": per_asset_df["asset"].nunique()
        })

    return {
        "summary": pd.DataFrame(results),
        "per_asset_ic": pd.concat(per_asset_list, ignore_index=True),
        "per_date_ic": pd.concat(per_date_list, ignore_index=True),
    }

In [ ]:
spot_df["us_hy_ig_spd"] = spot_df["us_hy_spread"]/spot_df["us_ig_spread"]
spot_df["us_10y_2y_spd"] = spot_df["us_10y_yield"] - spot_df["us_2y_yield"]
spot_df["us_cn_10y_spd"] = spot_df["us_10y_yield"] - spot_df["china_10y_yield"]
spot_df["us_de_10y_spd"] = spot_df["us_10y_yield"] - spot_df["german_10y_yield"]
spot_df["cn_de_10y_spd"] = spot_df["china_10y_yield"] - spot_df["german_10y_yield"]
spot_df['china_m2_m1_spd'] = spot_df['china_m2_growth_yoy'] - spot_df['china_m1_growth_yoy'] 

In [ ]:
spot_df[["china_soybean_crush_margin"]].dropna()

In [ ]:
feature_name = "china_soybean_crush_margin"
f1_px.columns.name = "asset"
feature_df = spot_df[[feature_name]].dropna().copy()
feature_df = feature_df- feature_df.rolling(20).mean() # - feature_df.rolling(8).mean() #.diff() #.pct_change(20) #.diff() # - feature_df.rolling(20).mean() #.diff() #.ewm(10).mean()
panel_df = build_feature_return_panel(f1_px, feature_df, feature_name, forward_windows=[1, 2, 3], beta_window=0)


In [ ]:
res = analyze_feature_ic_ir(panel_df, method="spearman")
summary_df = res['summary']
per_asset_df = res['per_asset_ic']
per_date_df = res['per_date_ic']
display(summary_df)
display(pd.pivot_table(per_asset_df, index='horizon', columns='asset', values='ic', aggfunc='last'))
display(pd.pivot_table(per_asset_df, index='horizon', columns='asset', values='pval', aggfunc='last'))

In [ ]:
#iplot(pd.pivot_table(per_date_df, index='date', columns='horizon', values='ic', aggfunc='last'))
iplot(pd.pivot_table(per_date_df, index='date', columns='horizon', values='ic', aggfunc='last').rolling(20).mean())

In [ ]:
import seaborn as sns
sns.heatmap(per_asset_df.pivot(index="asset", columns="horizon", values="ic"), cmap="RdBu_r", center=0)
plt.show()

In [ ]:
per_asset_df.pivot(index="asset", columns="horizon", values="ic")

In [ ]:
iplot(spot_df[['china_policy_uncertainty_index', 'us_policy_uncertainty_index']])

In [ ]:
cutoff='2018-01-01'
shift_holdings = 1
signal_cap = [-2, 2]
chg_func = 'diff'
bullish = False
vol_win = 20
by_asset = False

signal_func = 'zscore'
param_rng = [16, 24, 1]
#feature = "coke_inv_3ports"
#feature = "sinv"
feature = 'china_policy_uncertainty_index'
#feature = "pb_scrap_diff"
#feature = 'io_inv_45ports'
#feature = "sinv" #"ckc_inv_110washery"
#feature = 'scrap_invdays_300mill'
#feature = 'rebar_billet'
#feature = 'steel_total_stockdays'
#feature = 'steel_social_inv'
#feature='hrc_margin_sb'
freq=''
signal_df = pd.DataFrame(index=df_pxchg.index)

for asset in empiric_assets:
    #feature_ts = df_pxchg[asset].cumsum()
    #feature_ts = df[(asset+'c1', 'close')].dropna() #.pct_change() 
    if by_asset:
        asset_feature = f"{asset}_{feature}"
    else:
        asset_feature = feature
    feature_ts =spot_df[asset_feature].dropna() #.rolling(100).sum()
    #feature_ts = feature_ts.ewm(1).mean()
    #feature_ts = yoy_generic(feature_ts, label_func=calendar_label, group_col='label_day', func=chg_func)[asset_feature]    
    #feature_ts = yoy_generic(feature_ts, label_func=lunar_label, group_col='label_day', func=chg_func)[asset_feature]    
    signal_ts = calc_conv_signal(feature_ts, signal_func=signal_func, param_rng=param_rng, signal_cap=signal_cap, vol_win=vol_win) 
    #signal_ts = feature_ts
    #signal_ts = 2*signal_ts - 1*signal_ts.shift(1)
    #signal_ts = np.sign(signal_ts)    
    #signal_ts = conv_ewm(feature_ts, [2, 6, 2], [8, 32, 4], vol_win=60)        
    #signal_ts = ewmac(feature_ts, 8, 16, vol_win=0)
    #signal_ts = pd.Series(1, index=df_pxchg.index)
    #signal_ts.loc[signal_ts.index.month.isin([1, 2, 3, 4, 5, 6, 7, 12])] = 1
    #signal_ts.loc[signal_ts.index.day.isin(range(16, 32))] += 1
    #signal_ts = hlratio(feature_ts, 60)
    #signal_ts = signal_hysteresis(signal_ts, 0.7, 0.2)
    #signal_ts = seasonal_score(feature_ts.to_frame(), backward=10, forward=10, rolling_years=5, min_obs=10)
    #signal_ts = create_holiday_window_series(df.index, opt_expiries, 1, 3) # - create_holiday_window_series(df.index, opt_expiries, 3, 4)
    if not bullish:
        signal_ts = -signal_ts    
    
    signal_ts = signal_ts.reindex(index=cdates).ffill().reindex(index=df_pxchg.index)
    #signal_ts = signal_ts.rolling(2).mean()
    #signal_ts = signal_ts.ewm(1).mean()
    #signal_ts = signal_ts.shift(2)    
    #signal_ts.loc[signal_ts.index.month.isin([1, 2])] = 0
    #signal_ts = signal_hump(signal_ts, 0.2)
    if ('-' in asset) or ('=' in asset):
        sub_assets = re.split('[-=]', asset)
        if '=' in asset:
            spd_win = 20
            spd_vol_win = 20
            spd_px = df_pxchg[sub_assets[0]]/df_pxchg[sub_assets[0]].rolling(spd_win).std() - df_pxchg[sub_assets[1]]/df_pxchg[sub_assets[1]].rolling(spd_win).std()
            spd_vol = spd_px.rolling(spd_vol_win).std()
        else:
            spd_vol = pd.Series(1, index=signal_ts.index)
        if sub_assets[0] in signal_df.columns:
            signal_df[sub_assets[0]] += signal_ts/spd_vol
        else:
            signal_df[sub_assets[0]] = signal_ts/spd_vol
        if sub_assets[1] in signal_df.columns:
            signal_df[sub_assets[1]] -= signal_ts/spd_vol
        else:
            signal_df[sub_assets[1]] = -signal_ts/spd_vol    
    else:
        signal_df[asset] = signal_ts

#signal_df = xs_demean(signal_df)
#signal_df = signal_df + xs_demean(signal_df) * 0.5
#signal_df = signal_buffer(signal_df, 0.2)

holding = generate_holding_from_signal(signal_df, vol_df, risk_scaling=1.0, asset_scaling=False)

bt_metrics = MetricsBase(holdings=holding[signal_df.columns][cutoff:],
                         returns=df_pxchg[signal_df.columns][cutoff:], 
                         shift_holdings=shift_holdings)
trading_cost = dict([(asset, 0.6e-4) if asset in ["T", "TF"] else (asset, 2e-4) for asset in holding.columns])

bt_metrics_w_cost = MetricsBase(holdings=holding[signal_df.columns][cutoff:],
                                returns=df_pxchg[signal_df.columns][cutoff:], 
                                shift_holdings=shift_holdings,
                                cost_dict=trading_cost)

pnl_stats = bt_metrics.calculate_pnl_stats(shift=0, use_log_returns=False, tenors=pnl_tenors, perf_metrics=['sharpe', 'std', 'sortino', 'calmar'])
perf_stats = transform_output(pnl_stats)

pnl_stats_w_cost = bt_metrics_w_cost.calculate_pnl_stats(shift=0, use_log_returns=False, tenors=pnl_tenors, perf_metrics=['sharpe', 'std', 'sortino', 'calmar'])
perf_stats_w_cost = transform_output(pnl_stats_w_cost)
print("SR after cost:\n", perf_stats_w_cost)
print(pnl_stats_w_cost['asset_sharpe_stats'])
print("Turnover: \n%s\nPNL per trade:\n%s\n" % (pnl_stats_w_cost['turnover'], pnl_stats_w_cost['pnl_per_trade']))

print("SR before cost:\n", perf_stats)
print(pnl_stats['asset_sharpe_stats'])
print("Turnover: \n%s\nPNL per trade:\n%s\n" % (pnl_stats['turnover'], pnl_stats['pnl_per_trade']))

iplot(pnl_stats_w_cost['portfolio_cumpnl'], title='portfolio pnl with cost')
iplot(pnl_stats_w_cost['asset_cumpnl'], title='asset pnl with cost')

iplot(pnl_stats['portfolio_cumpnl'], title='portfolio pnl wo cost')
iplot(pnl_stats['asset_cumpnl'], title='asset pnl wo cost')